In [ ]:
%matplotlib inline
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, pauli_error
from qiskit.visualization import plot_histogram
import matplotlib.pyplot as plt

# =====================================================
# 4-BIT VEDRAL ADDER
# TEST CASE:
# 1000 + 1000 = 10000
# =====================================================

qc = QuantumCircuit(13, 5)

# =====================================================
# QUBIT MAPPING
#
# q0  = c0
# q1  = a0
# q2  = b0
# q3  = c1
# q4  = a1
# q5  = b1
# q6  = c2
# q7  = a2
# q8  = b2
# q9  = c3
# q10 = a3
# q11 = b3
# q12 = c4
# =====================================================

# =====================================================
# INPUT:
# A = 1000
# B = 1000
#
# Only MSB bits become 1
# =====================================================
qc.x(10)
qc.x(11)

qc.barrier()

qc.ccx(1,2,3)
qc.ccx(4,5,6)
qc.ccx(7,8,9)
qc.ccx(10,11,12)

qc.cx(1,2)
qc.cx(4,5)
qc.cx(7,8)
qc.cx(10,11)

qc.ccx(0,2,3)
qc.ccx(3,5,6)
qc.ccx(6,8,9)
qc.ccx(9,11,12)

qc.cx(10,11)
qc.cx(10,11)
qc.cx(9,11)

qc.ccx(6,8,9)
qc.cx(7,8)
qc.ccx(7,8,9)
qc.cx(7,8)
qc.cx(6,8)

qc.ccx(3,5,6)
qc.cx(4,5)
qc.ccx(4,5,6)
qc.cx(4,5)
qc.cx(3,5)

qc.ccx(0,2,3)
qc.cx(1,2)
qc.ccx(1,2,3)
qc.cx(1,2)
qc.cx(0,2)

# =====================================================
# MEASUREMENTS
# =====================================================


qc.measure(2,0);
qc.measure(5,1);
qc.measure(8,2);
qc.measure(11,3);
qc.measure(12,4);




# ====================================================
# DRAW CIRCUIT
# ====================================================

print("\nCircuit Diagram:\n")
qc.draw("mpl")

# ====================================================
# BITFLIP NOISE MODEL
# ====================================================

p = 0.01

# ----------------------------------------------------
# 1-Qubit Bitflip Error
# ----------------------------------------------------

error_1 = pauli_error([
    ('X', p),
    ('I', 1-p)
])

# ----------------------------------------------------
# 2-Qubit Error
# E ⊗ E
# ----------------------------------------------------

error_2 = error_1.tensor(error_1)

# ----------------------------------------------------
# 3-Qubit Error
# E ⊗ E ⊗ E
# ----------------------------------------------------

error_3 = error_1.tensor(error_1).tensor(error_1)

# ====================================================
# CREATE NOISE MODEL
# ====================================================

noise_model = NoiseModel()

# ----------------------------------------------------
# Apply noise to X gates
# ----------------------------------------------------

noise_model.add_all_qubit_quantum_error(
    error_1,
    ['x']
)

# ----------------------------------------------------
# Apply noise to CX gates
# ----------------------------------------------------

noise_model.add_all_qubit_quantum_error(
    error_2,
    ['cx']
)

# ----------------------------------------------------
# Apply noise to CCX gates
# ----------------------------------------------------

noise_model.add_all_qubit_quantum_error(
    error_3,
    ['ccx']
)

# ====================================================
# CREATE NOISY SIMULATOR
# ====================================================

sim = AerSimulator(noise_model=noise_model)

# ====================================================
# TRANSPILE CIRCUIT
# ====================================================

compiled = transpile(qc, sim)

# ====================================================
# RUN SIMULATION
# ====================================================

shots = 10000

result = sim.run(
    compiled,
    shots=shots
).result()

counts = result.get_counts()

# ====================================================
# SHOW COUNTS
# ====================================================

print("\nMeasurement Counts:\n")
print(counts)

# ====================================================
# PLOT HISTOGRAM
# ====================================================

plot_histogram(counts)
plt.show()

# ====================================================
# ERROR RATE CALCULATION
# ====================================================

# Expected:
# 1000 + 1000 = 10000

# ====================================================
# ERROR METRICS
# ====================================================

correct_output = '10000'
correct_decimal = 16

D = 30  # Maximum possible output for 4-bit addition

# ----------------------------------------------------
# ERROR RATE (ER)
# ----------------------------------------------------

correct_counts = counts.get(correct_output, 0)

ER = 1 - (correct_counts / shots)

# ----------------------------------------------------
# CALCULATE ED, NMED, MRED
# ----------------------------------------------------

total_ED = 0
total_relative_ED = 0

for output, freq in counts.items():

    # Convert binary string to decimal
    noisy_decimal = int(output, 2)

    # Error Distance
    ED = abs(correct_decimal - noisy_decimal)

    # Accumulate weighted ED
    total_ED += ED * freq

    # Avoid divide-by-zero
    if correct_decimal != 0:
        total_relative_ED += (ED / correct_decimal) * freq

# ----------------------------------------------------
# MEAN ERROR DISTANCE
# ----------------------------------------------------

mean_ED = total_ED / shots

# ----------------------------------------------------
# NMED
# ----------------------------------------------------

NMED = mean_ED / D

# ----------------------------------------------------
# MRED
# ----------------------------------------------------

MRED = total_relative_ED / shots

# ====================================================
# DISPLAY RESULTS
# ====================================================

print("\n===================================")
print("RESULTS")
print("===================================")

print(f"\nCorrect Output: {correct_output}")

print(f"\nCorrect Counts: {correct_counts}")

print(f"\nError Rate (ER): {ER}")

print(f"\nMean Error Distance: {mean_ED}")

print(f"\nNMED: {NMED}")

print(f"\nMRED: {MRED}")

print(f"\nSuccess Rate: {correct_counts/shots}")

# ====================================================
# OPTIONAL:
# DISPLAY SUCCESS RATE
# ====================================================

success_rate = correct_counts / shots

print("\nSuccess Rate:")
print(success_rate)